# MongoDB Integration

In this notebook, the processed parquet outputs generated by the Spark processing notebook are loaded into MongoDB.
The goal is to store the analytical results in separate MongoDB collections, create indexes, and run example queries.

# 01 Import Libraries

In this section, we import the Python libraries needed for MongoDB connection, data handling, and performance testing.

In [21]:
from pymongo import MongoClient
import pandas as pd
import os
import time


# 02 Connect to MongoDB

Here, we create a connection to the local MongoDB server and define the database and collection that will be used in this project.

In [22]:
client = MongoClient("mongodb://localhost:27017/")
db = client["olist_ecommerce"]
customer_spending_col = db["customer_spending"]
top_categories_col = db["top_categories"]
monthly_trends_col = db["monthly_trends"]
customer_segments_col = db["customer_segments"]

print("Connected to MongoDB")


Connected to MongoDB


# 03 Load Processed Data

In this section, we load the processed parquet outputs generated by the Spark processing notebook.
This allows the MongoDB notebook to continue from the previous pipeline stage instead of reading the raw CSV files again.

In [23]:
customer_spending_path = "../data/processed/customer_spending.parquet"
top_categories_path = "../data/processed/top_categories.parquet"
monthly_trends_path = "../data/processed/monthly_trends.parquet"
customer_segments_path = "../data/processed/customer_segments.parquet"

print("Current working directory:", os.getcwd())
print("customer_spending exists:", os.path.exists(customer_spending_path))
print("top_categories exists:", os.path.exists(top_categories_path))
print("monthly_trends exists:", os.path.exists(monthly_trends_path))
print("customer_segments exists:", os.path.exists(customer_segments_path))

customer_spending = pd.read_parquet(customer_spending_path)
top_categories = pd.read_parquet(top_categories_path)
monthly_trends = pd.read_parquet(monthly_trends_path)
customer_segments = pd.read_parquet(customer_segments_path)


Current working directory: /home/topi/projektit/data_science/big_data/BigData-Ecommerce-Analysis/notebooks
customer_spending exists: True
top_categories exists: True
monthly_trends exists: True
customer_segments exists: True


# 04 Insert into MongoDB

In this step, each processed pandas DataFrame is converted into JSON-like documents and inserted into its own MongoDB collection.

Each document represents an aggregated analytical record (such as customer spending, category performance, monthly trends, or customer segment).

In [24]:
def insert_collection(col, df, name):
    col.delete_many({})
    result = col.insert_many(df.to_dict("records"))
    print(f"{name}: inserted {len(result.inserted_ids)} documents")

insert_collection(customer_spending_col, customer_spending, "customer_spending")
insert_collection(top_categories_col, top_categories, "top_categories")
insert_collection(monthly_trends_col, monthly_trends, "monthly_trends")
insert_collection(customer_segments_col, customer_segments, "customer_segments")


customer_spending: inserted 95420 documents
top_categories: inserted 74 documents
monthly_trends: inserted 24 documents
customer_segments: inserted 95420 documents


# 05 Create Indexes

Indexes are created to improve query performance. The selected fields are commonly used in filtering, sorting, and aggregation operations, such as customer identifiers, category names, and time-based fields.

In [25]:
customer_spending_col.create_index("customer_unique_id")
customer_spending_col.create_index("total_spent")
top_categories_col.create_index("product_category_name")
top_categories_col.create_index("total_revenue")
monthly_trends_col.create_index([("year", 1), ("month", 1)])
customer_segments_col.create_index("customer_unique_id")
customer_segments_col.create_index("segment")


'segment_1'

# 06 Run Queries

This section contains the main MongoDB queries used in the project.
The queries are used to count documents, filter orders by state, and aggregate sales by state.

## Query 1: Top Customers by Spending

This query retrieves the top customers based on total spending.

In [26]:
top_customers = list(
    customer_spending_col.find({}, {"_id": 0})
    .sort("total_spent", -1)
    .limit(10)
)

top_customers


[{'customer_unique_id': '0a0a92112bd4c708ca5fde585afaa872',
 'total_spent': 13664.08,
 'total_orders': 1},
{'customer_unique_id': 'da122df9eeddfedc1dc1f5349a1a690c',
 'total_spent': 7571.63,
 'total_orders': 2},
{'customer_unique_id': '763c8b1c9c68a0229c42c9fc6f662b93',
 'total_spent': 7274.88,
 'total_orders': 1},
{'customer_unique_id': 'dc4802a71eae9be1dd28f5d788ceb526',
 'total_spent': 6929.31,
 'total_orders': 1},
{'customer_unique_id': '459bef486812aa25204be022145caa62',
 'total_spent': 6922.21,
 'total_orders': 1},
{'customer_unique_id': 'ff4159b92c40ebe40454e3e6a7c35ed6',
 'total_spent': 6726.66,
 'total_orders': 1},
{'customer_unique_id': '4007669dec559734d6f53e029e360987',
 'total_spent': 6081.54,
 'total_orders': 1},
{'customer_unique_id': '5d0a2980b292d049061542014e8960bf',
 'total_spent': 4809.44,
 'total_orders': 1},
{'customer_unique_id': 'eebb5dda148d3893cdaf5b5ca3040ccb',
 'total_spent': 4764.34,
 'total_orders': 1},
{'customer_unique_id': '48e1ac109decbb87765a3eade6854

## Query 2: Top Revenue Categories

This query retrieves product categories with the highest total revenue.

In [27]:
top_revenue_categories = list(
    top_categories_col.find({}, {"_id": 0})
    .sort("total_revenue", -1)
    .limit(10)
)

top_revenue_categories


[{'product_category_name': 'beleza_saude',
 'total_revenue': 1258681.34,
 'total_items_sold': 9670},
{'product_category_name': 'relogios_presentes',
 'total_revenue': 1205005.68,
 'total_items_sold': 5991},
{'product_category_name': 'cama_mesa_banho',
 'total_revenue': 1036988.68,
 'total_items_sold': 11115},
{'product_category_name': 'esporte_lazer',
 'total_revenue': 988048.97,
 'total_items_sold': 8641},
{'product_category_name': 'informatica_acessorios',
 'total_revenue': 911954.32,
 'total_items_sold': 7827},
{'product_category_name': 'moveis_decoracao',
 'total_revenue': 729762.49,
 'total_items_sold': 8334},
{'product_category_name': 'cool_stuff',
 'total_revenue': 635290.85,
 'total_items_sold': 3796},
{'product_category_name': 'utilidades_domesticas',
 'total_revenue': 632248.66,
 'total_items_sold': 6964},
{'product_category_name': 'automotivo',
 'total_revenue': 592720.11,
 'total_items_sold': 4235},
{'product_category_name': 'ferramentas_jardim',
 'total_revenue': 485256.46

## Query 3: Monthly Revenue Trends

This query retrieves monthly order counts and revenue over time.

In [28]:
monthly_revenue = list(
    monthly_trends_col.find({}, {"_id": 0})
    .sort([("year", 1), ("month", 1)])
)

monthly_revenue[:10]


[{'year': 2016, 'month': 9, 'total_orders': 3, 'monthly_revenue': 211.29},
{'year': 2016, 'month': 10, 'total_orders': 308, 'monthly_revenue': 56884.89},
{'year': 2016, 'month': 12, 'total_orders': 1, 'monthly_revenue': 19.62},
{'year': 2017, 'month': 1, 'total_orders': 789, 'monthly_revenue': 137251.79},
{'year': 2017,
 'month': 2,
 'total_orders': 1733,
 'monthly_revenue': 286340.87},
{'year': 2017,
 'month': 3,
 'total_orders': 2641,
 'monthly_revenue': 432087.03},
{'year': 2017,
 'month': 4,
 'total_orders': 2391,
 'monthly_revenue': 412562.01},
{'year': 2017,
 'month': 5,
 'total_orders': 3660,
 'monthly_revenue': 586406.28},
{'year': 2017,
 'month': 6,
 'total_orders': 3217,
 'monthly_revenue': 503138.27},
{'year': 2017,
 'month': 7,
 'total_orders': 3969,
 'monthly_revenue': 585076.47}]

## Query 4: Customer Segment Distribution

This aggregation groups customers by segment and counts how many belong to each segment.

In [29]:
segment_distribution = list(
    customer_segments_col.aggregate([
        {"$group": {"_id": "$segment", "customers": {"$sum": 1}}},
        {"$sort": {"customers": -1}}
    ])
)

segment_distribution


[{'_id': 'Medium Spender', 'customers': 46878},
{'_id': 'Low Spender', 'customers': 44117},
{'_id': 'High Spender', 'customers': 4425}]

# 07 Performance Test

In this section, we measure the execution time of a MongoDB query. This gives a simple example of query performance after indexing.

In [30]:
start = time.time()
list(
    customer_spending_col.find({}, {"_id": 0})
    .sort("total_spent", -1)
    .limit(100)
)
end = time.time()

print("Query time:", end - start)


Query time: 0.0010259151458740234


# Conclusion

In this notebook, the processed parquet outputs generated by the Spark processing notebook were loaded into MongoDB. The analytical datasets were stored in separate collections, indexed, and queried for analysis.

The notebook demonstrated querying top customers, category performance, monthly trends, and customer segmentation. This shows how MongoDB can be used as the storage and query layer of the Spark + MongoDB pipeline.